# AutoCutClips — Colab video queue + YouTube upload

Paste a list of YouTube links, get vertical Shorts for every video, then upload the best clips to your channel.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

Run the sections in order:
1. **Setup** — clone, install, optional Google Drive persistence
2. **Secrets & cookies** — API keys, `cookies.txt` for downloading, YouTube token for uploading
3. **Video queue** — paste links and render
4. **Review** — best clips across the queue
5. **Upload to YouTube**
6. **Learn** — feed your channel's results back into clip selection

GitHub: [pdrajan0x/AutoCutClips](https://github.com/pdrajan0x/AutoCutClips)

# 1. Setup

In [ ]:
# Clone the project into /content
!rm -rf ./* ./.[!.]*
!git clone https://github.com/pdrajan0x/AutoCutClips.git .

In [ ]:
# System packages (FFmpeg + Deno, which yt-dlp uses for YouTube's JS challenges)
import os

!apt-get -qq update
!apt-get -qq install -y ffmpeg
!curl -fsSL https://deno.land/install.sh | sh -s -- -y

os.environ["PATH"] += ":/root/.deno/bin"

In [ ]:
%%capture
!pip install -r requirements.txt gdown

In [ ]:
# Optional: keep outputs on Google Drive, so the queue can RESUME after a Colab
# disconnect (finished videos are skipped) and the upload history survives.
USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/AutoCutClips"

import os
import shutil

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(f"{DRIVE_DIR}/outputs", exist_ok=True)
    if os.path.isdir("outputs") and not os.path.islink("outputs"):
        shutil.rmtree("outputs")
    if not os.path.exists("outputs"):
        os.symlink(f"{DRIVE_DIR}/outputs", "outputs")
    print("✅ outputs/ ->", os.path.realpath("outputs"))
else:
    os.makedirs("outputs", exist_ok=True)

# 2. Secrets & cookies

Add these in the 🔑 **Secrets** panel (left sidebar) and enable notebook access:

| Secret | Needed for |
|---|---|
| `GOOGLE_API_KEY` | **Required** — Gemini clip selection |
| `PEXELS_API_KEY` | Optional — B-roll |
| `HF_TOKEN` | Optional — split-screen / camera-switch podcast modes |
| `YT_COOKIES` | **Downloading** — full contents of your `cookies.txt` |
| `YOUTUBE_TOKEN_JSON` | **Uploading** — contents of `.credentials/youtube_token.json` |
| `YOUTUBE_CLIENT_SECRET_JSON` | Uploading, only to create the token inside Colab (section 5) |

**About cookies:** Colab's IPs usually get *"Sign in to confirm you're not a bot"* from YouTube.
Export `cookies.txt` (Netscape format) from a browser logged in to YouTube, e.g. with the
"Get cookies.txt LOCALLY" extension, using a private window you then close so the cookies stay valid.
Cookies are only used to **download** videos; a secondary Google account is safest.

**Cookies cannot upload to your channel.** YouTube only accepts uploads through the Data API with an
OAuth token — see section 5. If the cookies are too large for a secret, set `COOKIES_PATH` below to a
`cookies.txt` you uploaded to Colab or Drive.

In [ ]:
import os
from pathlib import Path

from google.colab import userdata


def secret(name):
    try:
        return (userdata.get(name) or "").strip()
    except Exception:
        return ""


# --- API keys -> .env (read by every pipeline run) ---
env = {
    "GOOGLE_API_KEY": secret("GOOGLE_API_KEY"),
    "PEXELS_API_KEY": secret("PEXELS_API_KEY"),
    "HF_TOKEN": secret("HF_TOKEN"),
}
if not env["GOOGLE_API_KEY"]:
    print("❌ GOOGLE_API_KEY secret is missing — clip selection will not run.")

# --- cookies.txt for yt-dlp ---
COOKIES_PATH = "/content/cookies.txt"  # or e.g. "/content/drive/MyDrive/cookies.txt"

cookies_secret = secret("YT_COOKIES")
if cookies_secret:
    Path(COOKIES_PATH).write_text(cookies_secret.replace("\\t", "\t") + "\n", encoding="utf-8")
    os.chmod(COOKIES_PATH, 0o600)

if os.path.isfile(COOKIES_PATH):
    lines = [l for l in Path(COOKIES_PATH).read_text(encoding="utf-8").splitlines()
             if l.strip() and not l.startswith("#")]
    if lines and all(len(l.split("\t")) >= 7 for l in lines):
        env["YTDLP_COOKIES_FILE"] = COOKIES_PATH
        print(f"🍪 Cookies ready: {COOKIES_PATH} ({len(lines)} entries)")
    else:
        print("⚠️ cookies.txt is not in Netscape format (tab-separated, 7 columns). "
              "If you pasted it into a secret and the tabs became spaces, upload the file instead.")
else:
    print("ℹ️ No cookies — downloads may hit YouTube's bot check on Colab.")

Path(".env").write_text("".join(f"{k}={v}\n" for k, v in env.items() if v), encoding="utf-8")
os.environ.update({k: v for k, v in env.items() if v})

# --- YouTube upload credentials (optional) ---
os.makedirs(".credentials", exist_ok=True)
token_json = secret("YOUTUBE_TOKEN_JSON")
if token_json:
    Path(".credentials/youtube_token.json").write_text(token_json, encoding="utf-8")
    print("🔐 YouTube token written to .credentials/youtube_token.json")

client_secret_json = secret("YOUTUBE_CLIENT_SECRET_JSON")
if client_secret_json:
    Path(".credentials/client_secret.json").write_text(client_secret_json, encoding="utf-8")

print("✅ .env written")

# 3. Video queue

Paste one YouTube link per line. The queue:
- renders each video into its own folder `outputs/queue/<video-id>/`
- records progress in `outputs/queue/queue_state.json`, so re-running this cell **skips finished videos**
- deletes each source video after its clips render (disk space), unless `KEEP_SOURCE = True`
- writes `outputs/queue/queue_manifest.json` — every clip from every video, best first

In [ ]:
LINKS = """
https://www.youtube.com/watch?v=Ip6pCjWp4lk
# https://www.youtube.com/watch?v=ANOTHER_ID
"""

CLIP_COUNT = 5            # max clips per video — the AI returns fewer if a video has fewer great moments
MIN_DURATION = 30         # shortest clip in seconds (many strong moments are 15-30s)
MAX_DURATION = 80         # longest clip in seconds (Shorts allow up to 180)
RATIO = "9:16"
FONT_STYLE = "DEFAULT"    # DEFAULT (Montserrat Black) | HORMOZI | CINEMATIC | STORYTELLER
LANGUAGE = "auto"         # spoken language: auto, or a code like hi, en, ta, te, mr, bn, es
CAPTION_SCRIPT = "latin"  # latin: Hindi etc. written in English letters ("mera naam ... hai"), never translated
                          # native: keep the original script (needs a caption font that supports it)
USE_BROLL = False         # stock footage over the speaker; usually weaker for talking-head clips
USE_BGM = True            # quiet, auto-ducked background music
BGM_DIR = ""              # your own music, e.g. f"{DRIVE_DIR}/bgm" with chill/ epic/ sad/ upbeat/ suspense/ subfolders
BGM_VOLUME = 0.12         # 0.05 = barely there, 0.25 = prominent
RETRY_FAILED = False
KEEP_SOURCE = False
EXTRA_FLAGS = ""          # any other clip flag, e.g. "--caption-case upper --hook-teaser --face-detector yolo"

import os
import shlex
from pathlib import Path

Path("links.txt").write_text(LINKS, encoding="utf-8")

flags = [
    "--links", "links.txt",
    "--clips", str(CLIP_COUNT),
    "--ratio", RATIO,
    "--font-style", FONT_STYLE,
    "--language", LANGUAGE,
    "--caption-script", CAPTION_SCRIPT,
    "--min-duration", str(MIN_DURATION),
    "--max-duration", str(MAX_DURATION),
]
if not USE_BROLL:
    flags.append("--no-broll")
if not USE_BGM:
    flags.append("--no-bgm")
else:
    flags += ["--bgm-volume", str(BGM_VOLUME)]
    if BGM_DIR:
        flags += ["--bgm-dir", BGM_DIR]
if RETRY_FAILED:
    flags.append("--retry-failed")
if KEEP_SOURCE:
    flags.append("--keep-source")
if os.environ.get("YTDLP_COOKIES_FILE"):
    flags += ["--cookies", os.environ["YTDLP_COOKIES_FILE"]]
flags += shlex.split(EXTRA_FLAGS)

cmd = "python -m app.cli queue " + " ".join(shlex.quote(f) for f in flags)
print(cmd)
!{cmd}

# 4. Review the clips

In [ ]:
import json
import os
import subprocess

from IPython.display import HTML, Video, display

manifest_path = "outputs/queue/queue_manifest.json"
clips = json.load(open(manifest_path, encoding="utf-8")) if os.path.exists(manifest_path) else []
print(f"{len(clips)} clip(s) in {manifest_path}\n")

for c in clips:
    status = c.get("youtube_upload_status") or "-"
    print(f"{c.get('viral_score', 0):>3}  {c.get('duration', 0):>5.1f}s  [{c.get('queue_video_key')}]  "
          f"{c.get('title', '')}  (upload: {status})")
    print(f"      {c.get('video_path')}")

# Preview the top clips inline. Colab cannot stream local files into the page,
# so each clip is shrunk to a small preview and embedded.
PREVIEW = 3
os.makedirs("/content/previews", exist_ok=True)
for i, c in enumerate(clips[:PREVIEW]):
    path = c.get("video_path", "")
    if not os.path.exists(path):
        continue
    preview = f"/content/previews/preview_{i}.mp4"
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", path, "-vf", "scale=-2:640",
         "-c:v", "libx264", "-preset", "veryfast", "-crf", "32", "-c:a", "aac", "-b:a", "64k", preview],
        check=False,
    )
    if os.path.exists(preview):
        display(HTML(f"<p><b>{c.get('viral_score', 0)} — {c.get('title', '')}</b></p>"))
        display(Video(preview, embed=True, width=270))

In [ ]:
# Optional: download every rendered clip as a zip
from google.colab import files

!cd outputs/queue && zip -qr /content/clips.zip . -i '*_ready.mp4' '*.json'
files.download("/content/clips.zip")

# 5. Upload to YouTube

Uploading uses the **YouTube Data API** with an OAuth token (cookies cannot upload).

**One-time token setup** (pick one):
- **On your computer:** put your OAuth *Desktop app* client JSON at `.credentials/client_secret.json`, run
  `python -m app.cli youtube-token generate`, then paste the resulting `.credentials/youtube_token.json`
  into the `YOUTUBE_TOKEN_JSON` secret. It refreshes itself, so you only do this once.
- **Inside Colab:** add the client JSON as the `YOUTUBE_CLIENT_SECRET_JSON` secret, re-run section 2, and run
  the next cell. Open the link, approve, then copy the `http://localhost...` address the browser fails to open
  and paste it back. Save the printed token into the `YOUTUBE_TOKEN_JSON` secret for next time.

Uploads go out as **private videos scheduled** to publish later, and the limits in `upload_safety.json`
apply (per run, per day, minimum hours between publishes, queue size). Already-uploaded clips are skipped.

In [ ]:
# Create the token inside Colab (skip if YOUTUBE_TOKEN_JSON is already set)
import os

from app.uploaders.youtube_token import generate_token_manual, verify_token

TOKEN = ".credentials/youtube_token.json"
if not os.path.exists(TOKEN):
    generate_token_manual(".credentials/client_secret.json", TOKEN)
    print("\nSave this as the YOUTUBE_TOKEN_JSON secret so you don't need to log in again:\n")
    print(open(TOKEN, encoding="utf-8").read())

verify_token(TOKEN)

In [ ]:
INTERVAL_HOURS = 24       # hours between scheduled publishes (upload_safety.json enforces a minimum)
REQUIRE_APPROVAL = False  # True = confirm each clip (type y/n in the box that appears)
TEST_MODE = False         # True = upload only the first pending clip
TIMEZONE = "Asia/Kolkata"

flags = [
    "--manifest-file", "outputs/queue/queue_manifest.json",
    "--updated-manifest", "outputs/queue/queue_manifest_uploaded.json",
    "--result-file", "outputs/queue/youtube_upload_results.json",
    "--interval-hours", str(INTERVAL_HOURS),
    "--tz-name", TIMEZONE,
]
if not REQUIRE_APPROVAL:
    flags.append("--no-approval")
if TEST_MODE:
    flags.append("--test-mode")

cmd = "python -m app.cli upload-youtube " + " ".join(flags)
print(cmd)
!{cmd}

# 6. Learn from your channel's results

Once uploads have been public for a couple of days, fetch their view counts. Every later queue run shows Gemini the
channel's best and weakest clips, so selection leans towards what works for **your** audience. It needs at least
6 public clips older than 48 hours; add `--no-channel-learning` to `EXTRA_FLAGS` to switch it off.

In [ ]:
!python -m app.cli learn-youtube